In [0]:
# Gold Layer: dim_date Dimension Table
# Domain: Cross-Domain Reference
# Source: silver.date
# Target: gold.dim_date
# Description: Derives SK_DateID surrogate key from DateValue (yyyyMMdd format),
#              filters to 1950-2020 range, and writes the calendar dimension
#              to the Gold star schema layer. Run_id carry-forwarded from Silver.

In [0]:
%run ../../02_common_utils/operations

In [0]:
from pyspark.sql.functions import col, date_format, current_timestamp, lit

# ---------------------------------------------------------------------------
# Configuration
# ---------------------------------------------------------------------------
CATALOG = "charles_schwab_retailbrokerage_dev_team_lemma"
SILVER_SCHEMA = "silver"
GOLD_SCHEMA = "gold"
DOMAIN = "CROSS"
BATCH = "ALL"

SOURCE_TABLE = f"{CATALOG}.{SILVER_SCHEMA}.date"
TARGET_TABLE = f"{CATALOG}.{GOLD_SCHEMA}.dim_date"

# Set catalog context for operations utility functions
spark.sql(f"USE CATALOG {CATALOG}")

# ---------------------------------------------------------------------------
# Retrieve run_id from Silver (carry-forward from upstream layer)
# ---------------------------------------------------------------------------
run_id_row = spark.sql(f"""
    SELECT DISTINCT _run_id
    FROM {SOURCE_TABLE}
    ORDER BY _run_id DESC
    LIMIT 1
""").collect()

RUN_ID = run_id_row[0]["_run_id"]
print(f"Reusing run_id from silver: {RUN_ID}")

# Start pipeline run
start_pipeline_run(spark, RUN_ID, BATCH)
log_pipeline_message(spark, RUN_ID, "INFO", "gold_dim_date", "Gold layer processing started for dim_date")

# ---------------------------------------------------------------------------
# Read from Silver layer
# ---------------------------------------------------------------------------
df_silver = spark.table(SOURCE_TABLE)
source_count = df_silver.count()
print(f"Source count (silver.date): {source_count}")

# ---------------------------------------------------------------------------
# Derive SK_DateID and apply Gold schema
#   - SK_DateID: INT derived from DateValue in yyyyMMdd format
#   - _load_ts: Gold layer load timestamp (new timestamp for gold)
#   - _run_id: Carry-forwarded from Silver for upstream lineage
#   - _batch: Carry-forwarded from Silver
# ---------------------------------------------------------------------------
df_gold = df_silver.select(
    date_format(col("DateValue"), "yyyyMMdd").cast("int").alias("SK_DateID"),
    col("DateValue"),
    col("DateDesc"),
    col("CalendarYearID"),
    col("CalendarYearDesc"),
    col("CalendarQtrID"),
    col("CalendarQtrDesc"),
    col("CalendarMonthID"),
    col("CalendarMonthDesc"),
    col("CalendarWeekID"),
    col("CalendarWeekDesc"),
    col("DayOfWeekNum"),
    col("DayOfWeekDesc"),
    col("FiscalYearID"),
    col("FiscalYearDesc"),
    col("FiscalQtrID"),
    col("FiscalQtrDesc"),
    col("HolidayFlag"),
    col("_batch"),
    current_timestamp().alias("_load_ts"),
    col("_run_id")
)

# ---------------------------------------------------------------------------
# Filter date range: 1950-01-01 to 2020-12-31
# ---------------------------------------------------------------------------
df_final = df_gold.filter(
    (col("DateValue") >= lit("1950-01-01")) &
    (col("DateValue") <= lit("2020-12-31"))
)

target_count = df_final.count()
print(f"Target count (gold.dim_date): {target_count}")
print(f"Expected: 25933")

# ---------------------------------------------------------------------------
# Write to Gold table (full overwrite)
# ---------------------------------------------------------------------------
df_final.write.format("delta").mode("overwrite").option(
    "overwriteSchema", "true"
).saveAsTable(TARGET_TABLE)

log_pipeline_message(spark, RUN_ID, "INFO", "gold_dim_date", f"gold.dim_date written successfully with {target_count} rows")
print(f"gold.dim_date written successfully: {target_count} rows")

In [0]:
# ---------------------------------------------------------------------------
# Reconciliation and Audit Logging
# ---------------------------------------------------------------------------
from pyspark.sql import Row

# Extract the carry-forwarded _run_id from the dataframe
carried_run_id = str(df_final.select("`_run_id`").first()[0])
print(f"Carry-forwarded _run_id: {carried_run_id}")

# Build reconciliation DataFrame for reporting
recon_data = [
    Row(
        run_id=carried_run_id, batch_id=BATCH, domain=DOMAIN,
        table_name="dim_date", source_layer="silver", target_layer="gold",
        source_count=source_count, target_count=target_count,
        variance=source_count - target_count,
        status="MATCH" if (source_count == target_count) else "MISMATCH"
    )
]

recon_df = spark.createDataFrame(recon_data)
display(recon_df)

# ---------------------------------------------------------------------------
# Log reconciliation to operations.pipeline_recon_results
# ---------------------------------------------------------------------------
log_pipeline_recon(
    spark, carried_run_id, BATCH, DOMAIN,
    table_name="dim_date",
    source_layer="silver",
    target_layer="gold",
    source_count=source_count,
    target_count=target_count
)

# Log audit event
log_audit_event(spark, carried_run_id, BATCH, "gold", "dim_date", "OVERWRITE", target_count)

# End pipeline run
end_pipeline_run(spark, carried_run_id, "SUCCESS")
log_pipeline_message(spark, carried_run_id, "INFO", "gold_dim_date", "Gold layer processing completed successfully for dim_date")

print(f"Pipeline run {carried_run_id} completed. Reconciliation logged to operations.")